### Sequential Pipeline

In [ ]:
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama

# Sample product reviews for testing
customer_name= 'Maleeha Khalid'
company_name= 'Amazon'

positive_review = """I absolutely love this coffee maker! It brews quickly and the coffee tastes amazing. 
The built-in grinder saves me so much time in the morning, and the programmable timer means 
I wake up to fresh coffee every day. Worth every penny and highly recommended to any coffee enthusiast."""

negative_review = """Disappointed with this laptop. It's constantly overheating after just 30 minutes of use, 
and the battery life is nowhere near the 8 hours advertised - I barely get 3 hours. 
The keyboard has already started sticking on several keys after just two weeks. Would not recommend to anyone."""



# Step 1: Define the prompt templates for each processing step
sentiment_template = """Analyze the sentiment of the following product review as positive, negative, or neutral.
Provide your analysis in the format: "SENTIMENT: [positive/negative/neutral]"

Review: {review}

Your analysis:
"""

summary_template = """Summarize the following product review into 3-5 key bullet points.
Each bullet point should be concise and capture an important aspect mentioned in the review.

Review: {review}
Sentiment: {sentiment}

Key points:
"""

response_template = """Write a helpful response to a customer based on their product review.
If the sentiment is positive, thank them for their feedback. If negative, express understanding 
and suggest a solution or next steps. Personalize based on the specific points they mentioned.
Always start with: Dear {customer_name}, 
and always end wit:
Best regards,
{company_name}

Review: {review}
Sentiment: {sentiment}
Key points: {summary}

Response to customer:
"""

# Create prompt templates for each step
sentiment_prompt = PromptTemplate(template = sentiment_template, 
                                  input_variables=['review'])

summary_prompt = PromptTemplate(template = summary_template, 
                                input_variables=['review', 'sentiment'])

response_prompt = PromptTemplate(template = response_template, 
                                 input_variables=['review', 'sentiment', 'summary', 'customer_name', 'company_name'])


# initialize llm 
model = "llama3.2:1b"

llm = ChatOllama(
    model=model,
    temperature=0
)

# PART 1: Traditional Chain Approach
# Create individual LLMChains for each step
sentiment_analysis_chain = LLMChain( llm = llm, prompt = sentiment_prompt, output_key='sentiment')
summary_chain = LLMChain( llm = llm, prompt = summary_prompt, output_key='summary')
response_chain = LLMChain( llm = llm, prompt = response_prompt, output_key='response')

# Create a SequentialChain to connect all steps
overall_traditional_chain = SequentialChain(
    chains = [sentiment_analysis_chain, summary_chain, response_chain],
    input_variables = ['review', 'customer_name', 'company_name'],
    output_variables = ['response']
)

# PART 2: LCEL Approach
# Create prompt templates using PromptTemplate.from_template for each step
sentiment_prompt_v2 = PromptTemplate.from_template(template = sentiment_template)

summary_prompt_v2 = PromptTemplate.from_template(template = summary_template)

response_prompt_v2 = PromptTemplate.from_template(template = response_template)


# Create individual chain components using the pipe operator (|)
sentiment_analysis_chain_v2 = sentiment_prompt_v2 | llm | StrOutputParser() 
summary_chain_v2 = summary_prompt_v2 | llm | StrOutputParser() 
response_chain_v2 = response_prompt_v2 | llm | StrOutputParser() 

# Connect the components using RunnablePassthrough.assign()
overall_chain = (
    RunnablePassthrough.assign(sentiment = sentiment_analysis_chain_v2)
        .assign(summary = summary_chain_v2)
        .assign(response = response_chain_v2)
)

# Test both implementations
def test_chains(review, customer_name, company_name):
    """Test both chain implementations with the given review"""
    print("\n" + "="*50)
    print(f"TESTING WITH REVIEW:\n{review[:100]}...\n")
    
    #print("TRADITIONAL CHAIN RESULTS:")
    # Run the traditional chain and print the results
    # result = overall_traditional_chain.run({
    #     'review': review, 
    #     'customer_name': customer_name, 
    #     'company_name': company_name
    #     })
    
    # print(result)

    print("\nLCEL CHAIN RESULTS:")
    # Run the LCEL chain and print the results
    result_v2 = overall_chain.invoke({
        'review': review, 
        'customer_name': customer_name, 
        'company_name': company_name
        })
    
    print(result_v2['response'])
    
    print("="*50)

# Run tests
test_chains(positive_review, customer_name, company_name)
test_chains(negative_review, customer_name, company_name)


TESTING WITH REVIEW:
I absolutely love this coffee maker! It brews quickly and the coffee tastes amazing. 
The built-in g...


LCEL CHAIN RESULTS:
Dear Maleeha Khalid,

Thank you so much for taking the time to share your wonderful experience with our coffee maker! We're thrilled to hear that it's brewing quickly and producing great-tasting coffee for you. The built-in grinder is indeed a game-changer, saving you so much time in the morning, and the programmable timer is a fantastic feature that ensures you always have a fresh cup of coffee every day. We're glad to hear that you're a coffee enthusiast and that you're enjoying the benefits of our coffee maker.

We appreciate your kind words and would like to thank you for your loyalty to Amazon. We're committed to providing the best products and services to our customers, and it's customers like you who make it all possible.

If you have any other questions or concerns, please don't hesitate to reach out. We'd be happy to help. In the m

### Sequential + Parellel Pipeline

In [38]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough
)

# Step 1: Define the prompt templates for each processing step
sentiment_template = """Analyze the sentiment of the following product review as positive, negative, or neutral.
Provide your analysis in the format: "SENTIMENT: [positive/negative/neutral]"

Review: {review}

Your analysis:
"""

# Here we removed ssentiment input from summary template so that summary and sentiment analysis can run in parellel
summary_template = """Summarize the following product review into 3-5 key bullet points.
Each bullet point should be concise and capture an important aspect mentioned in the review.

Review: {review}

Key points:
"""

response_template = """Write a helpful response to a customer based on their product review.
If the sentiment is positive, thank them for their feedback. If negative, express understanding 
and suggest a solution or next steps. Personalize based on the specific points they mentioned.
Always start with: Dear {customer_name}, 
and always end wit:
Best regards,
{company_name}

Review: {review}
Sentiment: {sentiment}
Key points: {summary}

Response to customer:
"""

# Create prompt templates using PromptTemplate.from_template for each step
sentiment_prompt = PromptTemplate.from_template(template = sentiment_template)

summary_prompt = PromptTemplate.from_template(template = summary_template)

response_prompt = PromptTemplate.from_template(template = response_template)


# Create individual chain components using the pipe operator (|)
sentiment_analysis_chain = sentiment_prompt | llm | StrOutputParser() 
summary_chain = summary_prompt | llm | StrOutputParser() 
response_chain = response_prompt | llm | StrOutputParser() 


parellel_chain = RunnableParallel(
    sentiment = sentiment_analysis_chain,
    summary = summary_chain,
    review = lambda x: x["review"], #extract review text from "review": {"review": "whatever."},
    customer_name=lambda x: x["customer_name"],
    company_name=lambda x: x["company_name"]
)


overall_chain = (
    parellel_chain
    | RunnablePassthrough.assign(
        response=response_chain
    )
)

test_chains(positive_review, customer_name, company_name)
test_chains(negative_review, customer_name, company_name)



TESTING WITH REVIEW:
I absolutely love this coffee maker! It brews quickly and the coffee tastes amazing. 
The built-in g...


LCEL CHAIN RESULTS:
Dear Maleeha Khalid,

Thank you so much for taking the time to share your wonderful experience with our coffee maker! We're thrilled to hear that it's brewing quickly and producing great-tasting coffee for you. The built-in grinder is indeed a game-changer, saving you so much time in the morning, and the programmable timer is a fantastic feature that ensures you wake up to a freshly brewed cup every day. We're glad to hear that you're a coffee enthusiast and that our product has met your expectations.

We appreciate your kind words about the coffee maker's quality and value. We're committed to providing our customers with the best possible products, and it's customers like you who make it all worthwhile.

If you have any other questions or concerns, please don't hesitate to reach out. We'd be happy to help. In the meantime, we'd like to off